# Deploying NVIDIA Nemotron-3-Super with vLLM

This notebook will walk you through how to run the `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B` model with vLLM.

[vLLM](https://docs.vllm.ai) is a fast and easy-to-use library for LLM inference and serving. 

For more details on the model [click here](https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16)

Prerequisites:
- NVIDIA GPU with recent drivers (>= 264 GB VRAM for BF16, >= 160 GB VRAM for FP8, >= 80 GB VRAM for NVFP4) and CUDA 12.x.
- Python 3.10+

Note: NVFP4 requires Blackwell architecture. This notebook includes B200-focused steps for NVFP4 setup and serving.

#### Launch on NVIDIA Brev
You can simplify the environment setup by using [NVIDIA Brev](https://developer.nvidia.com/brev). Click the button to launch this project on a Brev instance with the necessary dependencies pre-configured.

Once deployed, click on the "Open Notebook" button to get started with this guide. 

**For BF16 (4x H100):**

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-39czrpTlsxUjV3kJzRRbNt0bR7H)

**For FP8 (2x H100):**

[![Launch on Brev](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-3AjujUTh2ZqWdxbRrjlnEy2Qkvs)

## Install dependencies

For FP8, ensure CUDA toolkit and build tools are available in your environment before first run (required for kernel JIT on some setups):

```shell
sudo apt update
sudo apt install -y cuda-toolkit-12-8 ninja-build gcc g++ build-essential
export CUDA_HOME=/usr/local/cuda-12.8
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib64:${LD_LIBRARY_PATH}
```

FP8 may take longer on first startup due to kernel compile/autotune; later runs are usually faster from cache.

In [1]:
#If pip not found
!python -m ensurepip --default-pip

Looking in links: /tmp/tmpccruqxe6
Processing /tmp/tmpccruqxe6/pip-25.0.1-py3-none-any.whl


In [ ]:
%pip install -U vllm==0.17.1 torch==2.10.0 flashinfer-python==0.6.4 flashinfer-cubin==0.6.4 'nvidia-cutlass-dsl>=4.4.0.dev1' --extra-index-url https://download.pytorch.org/whl/cu128

## Verify GPU

Confirm CUDA is available and your GPU is visible to PyTorch.


In [3]:
# GPU environment check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")

CUDA available: True
Num GPUs: 4
GPU[0]: NVIDIA H100 PCIe
GPU[1]: NVIDIA H100 PCIe
GPU[2]: NVIDIA H100 PCIe
GPU[3]: NVIDIA H100 PCIe


## Load the model

Initialize the Nemotron model in vLLM with BF16, FP8, or NVFP4.

In [4]:
from vllm import LLM

# BF16 (4x H100)
model_id = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16"
# FP8 (2x H100)
# model_id = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8"
# NVFP4 (B200)
# model_id = "nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-NVFP4"

llm = LLM(
    model=model_id,
    trust_remote_code=True,
    dtype="auto",
    # Use TP=4 for BF16, TP=2 for FP8, and TP=1 for NVFP4.
    tensor_parallel_size=4,
    # tensor_parallel_size=2,
    # tensor_parallel_size=1,
    kv_cache_dtype="fp8",
    gpu_memory_utilization=0.9,
)

print("Model ready")

/home/shadeform/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 02-23 22:30:52 [utils.py:263] non-default args: {'trust_remote_code': True, 'kv_cache_dtype': 'fp8', 'tensor_parallel_size': 4, 'disable_log_stats': True, 'model': 'nvidia/nemotron-super-sft-020426'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
A new version of the following files was downloaded from https://huggingface.co/nvidia/nemotron-super-sft-020426:
- configuration_nemotron_h.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
2026-02-23 22:30:52,661	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 02-23 22:30:59 [model.py:530] Resolved architecture: NemotronHForCausalLM
INFO 02-23 22:30:59 [model.py:1545] Using max model len 262144
INFO 02-23 22:30:59 [cache.py:206] Using fp8 data type to store kv cache. It reduces the GPU memory footprint and boosts the performance. Meanwhile, it may cause accuracy drop without a proper scaling factor.
INFO 02-23 22:30:59 [scheduler.py:229] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 02-23 22:30:59 [config.py:543] Updating mamba_ssm_cache_dtype to 'float32' for NemotronH model
INFO 02-23 22:30:59 [config.py:476] Setting attention block size to 4160 tokens to ensure that attention page size is >= mamba page size.
INFO 02-23 22:30:59 [config.py:500] Padding mamba page size by 0.10% to ensure that mamba page size and attention page size are exactly equal.
INFO 02-23 22:30:59 [vllm.py:630] Asynchronous scheduling is enabled.
INFO 02-23 22:30:59 [vllm.py:637] Disabling NCCL for DP synchronization when using async scheduli

Loading safetensors checkpoint shards:   0% Completed | 0/50 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:   2% Completed | 1/50 [00:00<00:22,  2.20it/s]
Loading safetensors checkpoint shards:   4% Completed | 2/50 [00:00<00:22,  2.12it/s]
Loading safetensors checkpoint shards:   6% Completed | 3/50 [00:01<00:22,  2.13it/s]
Loading safetensors checkpoint shards:   8% Completed | 4/50 [00:01<00:21,  2.12it/s]
Loading safetensors checkpoint shards:  10% Completed | 5/50 [00:02<00:21,  2.09it/s]
Loading safetensors checkpoint shards:  12% Completed | 6/50 [00:02<00:21,  2.02it/s]
Loading safetensors checkpoint shards:  14% Completed | 7/50 [00:03<00:21,  2.01it/s]
Loading safetensors checkpoint shards:  16% Completed | 8/50 [00:03<00:20,  2.02it/s]
Loading safetensors checkpoint shards:  20% Completed | 10/50 [00:04<00:15,  2.62it/s]
Loading safetensors checkpoint shards:  22% Completed | 11/50 [00:04<00:16,  2.40it/s]
Loading safetensors checkpoint shards:  24% Completed | 12/5

(Worker_TP0 pid=14843) INFO 02-23 22:34:12 [default_loader.py:291] Loading weights took 23.97 seconds
(Worker_TP0 pid=14843) INFO 02-23 22:34:13 [gpu_model_runner.py:3905] Model loading took 56.94 GiB memory and 175.983706 seconds
(Worker_TP0 pid=14843) INFO 02-23 22:34:21 [backends.py:644] Using cache directory: /home/shadeform/.cache/vllm/torch_compile_cache/4852213ca2/rank_0_0/backbone for vLLM's torch.compile
(Worker_TP0 pid=14843) INFO 02-23 22:34:21 [backends.py:704] Dynamo bytecode transform time: 5.45 s
(Worker_TP3 pid=14846) INFO 02-23 22:34:24 [backends.py:261] Cache the graph of compile range (1, 16384) for later use
(Worker_TP0 pid=14843) INFO 02-23 22:34:24 [backends.py:261] Cache the graph of compile range (1, 16384) for later use
(Worker_TP2 pid=14845) INFO 02-23 22:34:24 [backends.py:261] Cache the graph of compile range (1, 16384) for later use
(Worker_TP1 pid=14844) INFO 02-23 22:34:28 [backends.py:261] Cache the graph of compile range (1, 16384) for later use


(Worker_TP2 pid=14845) /home/shadeform/.venv/lib/python3.12/site-packages/torch/_inductor/compile_fx.py:312: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
(Worker_TP2 pid=14845)   warnings.warn(
(Worker_TP0 pid=14843) /home/shadeform/.venv/lib/python3.12/site-packages/torch/_inductor/compile_fx.py:312: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
(Worker_TP0 pid=14843)   warnings.warn(
(Worker_TP3 pid=14846) /home/shadeform/.venv/lib/python3.12/site-packages/torch/_inductor/compile_fx.py:312: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
(Worker_TP3 pid=14846)   warning

(Worker_TP0 pid=14843) WARNING 02-23 22:34:31 [fused_moe.py:1090] Using default MoE config. Performance might be sub-optimal! Config file not found at /home/shadeform/.venv/lib/python3.12/site-packages/vllm/model_executor/layers/fused_moe/configs/E=512,N=672,device_name=NVIDIA_H100_PCIe.json
(Worker_TP0 pid=14843) INFO 02-23 22:35:03 [backends.py:278] Compiling a graph for compile range (1, 16384) takes 40.16 s
(Worker_TP0 pid=14843) INFO 02-23 22:35:03 [monitor.py:34] torch.compile takes 45.62 s in total
(Worker_TP0 pid=14843) INFO 02-23 22:35:05 [gpu_worker.py:358] Available KV cache memory: 7.94 GiB
(EngineCore_DP0 pid=14604) INFO 02-23 22:35:06 [kv_cache_utils.py:1305] GPU KV cache size: 690,560 tokens
(EngineCore_DP0 pid=14604) INFO 02-23 22:35:06 [kv_cache_utils.py:1310] Maximum concurrency for 262,144 tokens per request: 14.49x


(Worker_TP2 pid=14845) 2026-02-23 22:35:07,077 - INFO - autotuner.py:256 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(Worker_TP0 pid=14843) 2026-02-23 22:35:07,077 - INFO - autotuner.py:256 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(Worker_TP3 pid=14846) 2026-02-23 22:35:07,077 - INFO - autotuner.py:256 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(Worker_TP1 pid=14844) 2026-02-23 22:35:07,077 - INFO - autotuner.py:256 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(Worker_TP0 pid=14843) 2026-02-23 22:35:07,497 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process ends
(Worker_TP1 pid=14844) 2026-02-23 22:35:07,497 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process ends
(Worker_TP2 pid=14845) 2026-02-23 22:35:07,497 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process ends
(Worker_TP3 pid=14846) 2026-02-23 22:35:07,497 - INFO - autotuner.py:262 - flash

(Worker_TP0 pid=14843) INFO 02-23 22:35:26 [gpu_model_runner.py:4856] Graph capturing finished in 19 secs, took 1.10 GiB
(EngineCore_DP0 pid=14604) INFO 02-23 22:35:26 [core.py:273] init engine (profile, create kv cache, warmup model) took 71.27 seconds
(EngineCore_DP0 pid=14604) INFO 02-23 22:35:29 [vllm.py:630] Asynchronous scheduling is enabled.
INFO 02-23 22:35:29 [llm.py:347] Supported tasks: ['generate']
Model ready


## Generate responses

Generate text with vLLM using single, batched, and simple streaming examples.

### Single or batch prompts

Send one prompt or a list to run batched generation.

In [5]:
from transformers import AutoTokenizer
from vllm import SamplingParams

params = SamplingParams(temperature=0.6, max_tokens=200)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Single prompt
single_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": "Give me 3 bullet points about vLLM."}],
    tokenize=False,
    add_generation_prompt=True,
)
single = llm.generate([single_prompt], sampling_params=params)
print(single[0].outputs[0].text)

# Batch prompts
raw_prompts = [
    "Hello, my name is",
    "The capital of France is",
    "Explain quantum computing in simple terms:",
]
prompts = [
    tokenizer.apply_chat_template(
        [{"role": "user", "content": p}],
        tokenize=False,
        add_generation_prompt=True,
    )
    for p in raw_prompts
]
outputs = llm.generate(prompts, sampling_params=params)
for i, out in enumerate(outputs):
    print(f"\nPrompt {i+1}: {raw_prompts[i]!r}")
    print(out.outputs[0].text)

Processed prompts: 100%|██████████| 1/1 [00:23<00:00, 23.58s/it, est. speed input: 1.15 toks/s, output: 8.48 toks/s]


Okay, the user is asking for three bullet points about vLLM. Let me recall what I know about vLLM. It's a high-throughput inference library for large language models, right? Developed by the team at UC Berkeley. The main selling point is its efficiency in handling LLM inference, especially for serving.

First, I should focus on the key features that make vLLM stand out. The most notable is probably the PagedAttention mechanism. That's a big deal because it solves the memory fragmentation issue in traditional attention implementations. Traditional methods waste a lot of memory due to fragmentation, but PagedAttention uses a more efficient memory management approach. I should explain that clearly but concisely.

Next, the user might be interested in performance metrics. vLLM is known for its high throughput and low latency. I remember it achieves significantly higher throughput compared to other frameworks like Hugging Face Transformers or TensorRT-LLM. Maybe mention a specific benchmark

Processed prompts: 100%|██████████| 3/3 [00:02<00:00,  1.19it/s, est. speed input: 26.12 toks/s, output: 110.81 toks/s]


Prompt 1: 'Hello, my name is'
Hello! It's nice to meet you. How can I assist you today? 😊
</think>
Hello! It's nice to meet you. How can I assist you today? 😊

Prompt 2: 'The capital of France is'
The user asks: "The capital of France is". This is a simple factual question. The answer: Paris. Provide concise answer.
</think>
The capital of France is **Paris**.

Prompt 3: 'Explain quantum computing in simple terms:'
Okay, the user asked me to explain quantum computing in simple terms. Let me start by recalling what I know. Quantum computing is a complex topic, so I need to break it down without using too much jargon. The user probably isn't a physicist, so analogies and everyday examples would help.

First, I should compare it to classical computing since that's familiar. Classical computers use bits that are either 0 or 1. Quantum computers use qubits, which can be both 0 and 1 at the same time. That's superposition. But how to explain superposition simply? Maybe use a spinning coin—w

### Streamed generation

Print characters as they are produced.

In [6]:
def stream_like(prompt: str, llm: LLM, sampling_params: SamplingParams) -> None:
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
    )
    outputs = llm.generate([prompt], sampling_params=sampling_params)
    text = outputs[0].outputs[0].text
    print("Response:", end=" ")
    for ch in text:
        print(ch, end="", flush=True)
    print()

stream_like("Write a haiku about GPUs.", llm, SamplingParams(temperature=0.7, max_tokens=512))


Processed prompts: 100%|██████████| 1/1 [00:02<00:00,  2.23s/it, est. speed input: 10.78 toks/s, output: 95.67 toks/s]

Response: The user wants a haiku about GPUs. Haiku is 5-7-5 syllables. Must be about GPUs. Ensure correct syllable count. Provide a haiku. No extra text? Probably just the haiku. Let's craft: "Silicon hearts beat, / Crunching pixels, data streams / Speed of light awakes." Count syllables: "Silicon hearts beat" = Si-li-con (3) hearts(1) beat(1) =5. "Crunching pixels, data streams" = Crunch-ing(2) pix-els(2) da-ta(2) streams(1) =7. "Speed of light awakes" = Speed(1) of(1) light(1) a-wakes(2) =5? Actually "awakes" is 2 syllables (a-wakes). So total 5. Good. Provide that.
</think>
Silicon hearts beat,  
Crunching pixels, data streams,  
Speed of light awakes.


## OpenAI-compatible server

Serve the model via an OpenAI-compatible API using vLLM.

### Release GPU memory before server mode

This notebook used GPU memory for in-process inference.
Before starting `vllm serve`, run the cleanup cell below to free memory.

If server startup still runs out of memory, restart the kernel, then continue from this section.

Choose the variant you want to serve:
- BF16: `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16` on 4x H100
- FP8: `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8` on 2x H100
- NVFP4: `nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-NVFP4` on B200 

In [7]:
import gc
import torch

if "llm" in globals():
    del llm

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Cleanup complete. If server startup still OOMs, restart the kernel before proceeding.")

(Worker_TP0 pid=14843) INFO 02-23 22:41:05 [multiproc_executor.py:707] Parent process exited, terminating worker
(Worker_TP2 pid=14845) INFO 02-23 22:41:05 [multiproc_executor.py:707] Parent process exited, terminating worker
(Worker_TP3 pid=14846) INFO 02-23 22:41:05 [multiproc_executor.py:707] Parent process exited, terminating worker
(Worker_TP1 pid=14844) INFO 02-23 22:41:05 [multiproc_executor.py:707] Parent process exited, terminating worker
Cleanup complete. If server startup still OOMs, restart the kernel before proceeding.


### Prepare terminal environment

Use the same virtual environment where dependencies were installed.

For example, on Brev run:
```shell
source /home/shadeform/.venv/bin/activate
```

If you are not on Brev, replace this with your own virtual environment path.

### Start server

After cleanup (or after restart if needed), run this in a terminal.

#### BF16 (4x H100)

```shell
wget "https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16/resolve/main/super_v3_reasoning_parser.py"
```

```shell
vllm serve nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16 \
  --async-scheduling \
  --dtype auto \
  --kv-cache-dtype fp8 \
  --tensor-parallel-size 4 \
  --pipeline-parallel-size 1 \
  --data-parallel-size 1 \
  --swap-space 0 \
  --trust-remote-code \
  --gpu-memory-utilization 0.9 \
  --enable-chunked-prefill \
  --max-num-seqs 512 \
  --served-model-name nemotron \
  --host 0.0.0.0 \
  --port 5000 \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --reasoning-parser-plugin "./super_v3_reasoning_parser.py" \
  --reasoning-parser super_v3
```

#### FP8 (2x H100)

Note: the first FP8 server startup may take longer than BF16 while kernels are compiled/autotuned. This is expected.

```shell
wget "https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8/resolve/main/super_v3_reasoning_parser.py"
```

```shell
vllm serve nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-FP8 \
  --async-scheduling \
  --dtype auto \
  --kv-cache-dtype fp8 \
  --tensor-parallel-size 2 \
  --pipeline-parallel-size 1 \
  --data-parallel-size 1 \
  --swap-space 0 \
  --trust-remote-code \
  --attention-backend TRITON_ATTN \
  --gpu-memory-utilization 0.9 \
  --enable-chunked-prefill \
  --max-num-seqs 512 \
  --served-model-name nemotron \
  --host 0.0.0.0 \
  --port 5000 \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --reasoning-parser-plugin "./super_v3_reasoning_parser.py" \
  --reasoning-parser super_v3
```

#### NVFP4 (B200)

```shell
wget "https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-NVFP4/resolve/main/super_v3_reasoning_parser.py"
```

```shell
vllm serve nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-NVFP4 \
  --async-scheduling \
  --dtype auto \
  --kv-cache-dtype fp8 \
  --tensor-parallel-size 1 \
  --pipeline-parallel-size 1 \
  --data-parallel-size 1 \
  --swap-space 0 \
  --trust-remote-code \
  --attention-backend TRITON_ATTN \
  --gpu-memory-utilization 0.9 \
  --enable-chunked-prefill \
  --max-num-seqs 512 \
  --served-model-name nemotron \
  --host 0.0.0.0 \
  --port 5000 \
  --enable-auto-tool-choice \
  --tool-call-parser qwen3_coder \
  --reasoning-parser-plugin "./super_v3_reasoning_parser.py" \
  --reasoning-parser super_v3
```

Your server is now running!

### Use the API

Send chat and streaming requests to your vLLM server using the OpenAI-compatible client.

Note: The model supports two modes - Reasoning ON (default) vs OFF. This can be toggled by setting enable_thinking to False, as shown below.

In [8]:
# Client: Standard chat and streaming
from openai import OpenAI

client = OpenAI(base_url="http://127.0.0.1:5000/v1", api_key="null")

In [9]:
# Reasoning on (default)
print("Reasoning on")
resp = client.chat.completions.create(
    model="nemotron",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1,
    max_tokens=1024,
)
print("Reasoning:", resp.choices[0].message.reasoning_content, "\nContent:", resp.choices[0].message.content)
print("\n")
# Reasoning off
print("Reasoning off")
resp2 = client.chat.completions.create(
    model="nemotron",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Give me 3 interesting facts about vLLM."}
    ],
    temperature=0,
    max_tokens=256,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}}
)
print(resp2.choices[0].message.content)

Reasoning on
Reasoning: We need to produce a haiku (5-7-5 syllable poem) about GPUs. Should be creative. Probably mention silicon, compute, speed, cooling, etc. Count syllables.

Let's craft: "Silicon veins pulse" (5? Sil-i-con =3, veins=1, pulse=1 total 5). Good.

Second line 7 syllables: "Thunder of cores, rendering dreams". Count: Thun-der(2) of(1) cores(1)=4, ren-dering(2) =6, dreams(1)=7. Good.

Third line 5 syllables: "Cool fans whisper fast". Count: Cool(1) fans(1)=2, whis-per(2)=4, fast(1)=5. Done.

Check haiku format: 5-7-5. Good. Provide as answer.
 
Content: 
Silicon veins pulse  
Thunder of cores, rendering dreams  
Cool fans whisper fast


Reasoning off
Sure! Here are three interesting facts about **vLLM** (Virtual Large Language Model), a high-performance inference engine for large language models:

1. **Pioneering PagedAttention for Memory Efficiency**  
   vLLM introduced **PagedAttention**, a novel memory management technique inspired by virtual memory in operating sys

In [10]:
# Streaming chat completion
stream = client.chat.completions.create(
    model="nemotron",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What are the first 5 prime numbers?"}
    ],
    temperature=0.7,
    max_tokens=1024,
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta
    if delta and delta.content:
        print(delta.content, end="", flush=True)


The first five prime numbers are:

1. **2**  
2. **3**  
3. **5**  
4. **7**  
5. **11**

### Tool calling

Call functions using the OpenAI Tools schema and inspect returned tool_calls.

In [11]:
# Tool calling via OpenAI tools schema
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "calculate_tip",
            "parameters": {
                "type": "object",
                "properties": {
                    "bill_total": {
                        "type": "integer",
                        "description": "The total amount of the bill"
                    },
                    "tip_percentage": {
                        "type": "integer",
                        "description": "The percentage of tip to be applied"
                    }
                },
                "required": ["bill_total", "tip_percentage"]
            }
        }
    }
]

completion = client.chat.completions.create(
    model="nemotron",
    messages=[
        {"role": "system", "content": ""},
        {"role": "user", "content": "My bill is $50. What will be the amount for 15% tip?"}
    ],
    tools=TOOLS,
    temperature=0.6,
    top_p=0.95,
    max_tokens=512,
    stream=False
)

print(completion.choices[0].message.reasoning_content)
print(completion.choices[0].message.tool_calls)

The user wants to calculate a 15% tip on a $50 bill. I have access to the calculate_tip function which requires bill_total and tip_percentage as parameters. I need to call this function with bill_total = 50 and tip_percentage = 15. Let me do that.

[ChatCompletionMessageFunctionToolCall(id='chatcmpl-tool-a3926e1998a22969', function=Function(arguments='{"bill_total": 50, "tip_percentage": 15}', name='calculate_tip'), type='function')]


### Controlling Reasoning Budget

The `reasoning_budget` parameter allows you to limit the length of the model's reasoning trace. When the reasoning output reaches the specified token budget, the model will attempt to gracefully end the reasoning at the next newline character. 

If no newline is encountered within 500 tokens after reaching the budget threshold, the reasoning trace will be forcibly terminated at `reasoning_budget + 500` tokens to prevent excessive generation.


In [6]:
from typing import Any, Dict, List
import openai
from transformers import AutoTokenizer


class ThinkingBudgetClient:
    def __init__(self, base_url: str, api_key: str, tokenizer_name_or_path: str):
        self.base_url = base_url
        self.api_key = api_key
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name_or_path)
        self.client = openai.OpenAI(base_url=self.base_url, api_key=self.api_key)

    def chat_completion(
        self,
        model: str,
        messages: List[Dict[str, Any]],
        reasoning_budget: int = 512,
        max_tokens: int = 1024,
        **kwargs,
    ) -> Dict[str, Any]:
        assert (
            max_tokens > reasoning_budget
        ), f"reasoning_budget must be smaller than max_tokens. Given {max_tokens=} and {reasoning_budget=}"

        # 1. first call chat completion to get reasoning content
        response = self.client.chat.completions.create(
            model=model, 
            messages=messages, 
            max_tokens=reasoning_budget, 
            **kwargs
        )
    
        reasoning_content = response.choices[0].message.reasoning_content or ""
        
        if "</think>" not in reasoning_content:
            # reasoning content is too long, closed with a period (.)
            reasoning_content = f"{reasoning_content}.\n</think>\n\n"
        
        reasoning_tokens_used = len(
            self.tokenizer.encode(reasoning_content, add_special_tokens=False)
        )
        remaining_tokens = max_tokens - reasoning_tokens_used
        
        assert (
            remaining_tokens > 0
        ), f"remaining tokens must be positive. Given {remaining_tokens=}. Increase max_tokens or lower reasoning_budget."

        # 2. append reasoning content to messages and call completion
        messages.append({"role": "assistant", "content": reasoning_content})
        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            continue_final_message=True,
        )
        
        response = self.client.completions.create(
            model=model, 
            prompt=prompt, 
            max_tokens=remaining_tokens, 
            **kwargs
        )

        response_data = {
            "reasoning_content": reasoning_content.strip().strip("</think>").strip(),
            "content": response.choices[0].text,
            "finish_reason": response.choices[0].finish_reason,
        }
        return response_data

/home/shadeform/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
# Client
client = ThinkingBudgetClient(
    base_url="http://127.0.0.1:5000/v1",
    api_key="null",
    tokenizer_name_or_path="nvidia/NVIDIA-Nemotron-3-Super-120B-A12B-BF16"
)

In [8]:
resp = client.chat_completion(
    model="nemotron",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a haiku about GPUs."}
    ],
    temperature=1,
    max_tokens=256,
    reasoning_budget=32
)
print("Reasoning:", resp["reasoning_content"], "\nContent:", resp["content"])

Reasoning: The user asks: Write a haiku about GPUs. Haiku is 5-7-5 syllable poem. So need 5 syllables first line,. 
Content: 
Silicon hearts pulse  
in a cascade of glowing cores  
dreams rendered in math


## Cleanup and shutdown

To free resources after this notebook:

1. Release in-process model memory (run the next cell).
2. Stop the OpenAI-compatible `vllm serve` process in the terminal where it was started (`Ctrl+C`).
3. If needed, restart the kernel to ensure a clean state.

In [ ]:
import gc
import torch

# Release in-process model/tokenizer objects if present
for name in ("llm", "tokenizer"):
    if name in globals():
        del globals()[name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("In-process cleanup complete.")
print("If vllm serve is running in a terminal, stop it with Ctrl+C.")

In-process cleanup complete.
If vllm serve is running in a terminal, stop it with Ctrl+C.
